# Corrective RAG: a bounded recovery controller

## Northstar Cloud security-support scenario

A support engineer asks: **“How do I rotate the production API key for the payments integration?”** Your assistant may answer only from authorized runbooks. A nearby health-check document and a stale migration note are plausible distractors.

This notebook turns Corrective RAG from a slogan into an inspectable control system: retrieve → grade evidence → recover through a limited route → verify → answer or abstain. All executable examples are deterministic and credential-free.


## Learning objectives

After this notebook you can:

1. distinguish CRAG from standard RAG, Adaptive RAG, and Self-RAG;
2. implement an evidence grader and bounded route policy;
3. add reformulation and alternate-retriever recovery without expanding authorization;
4. preserve an attempt trace suitable for evaluation and incident review; and
5. choose production metrics, budgets, and safe terminal states.

**Primary source:** [Corrective Retrieval Augmented Generation](https://arxiv.org/abs/2401.15884). The paper proposes retrieval-quality evaluation, corrective retrieval actions, web-search augmentation for limited corpora, and decompose-then-recompose document processing.


## The control loop

```mermaid
flowchart TD
  Q[Authorized question] --> R[Primary retrieval]
  R --> E{Evidence grade}
  E -->|strong| B[Evidence bundle]
  E -->|ambiguous / weak| W[Bounded rewrite]
  W --> E2{Recovered?}
  E2 -->|strong| B
  E2 -->|weak| A[Approved alternate retrieval]
  A --> E3{Recovered?}
  E3 -->|strong| B
  E3 -->|weak or budget exhausted| X[Clarify / abstain / escalate]
  B --> V[Verify answer support]
  V --> O[Answer with citations + trace]
```

A corrective system is not allowed to “try harder” indefinitely. Every edge has an authorization rule, a retry/time budget, and an observable reason.


## 1 — Create a controlled corpus

The example uses lexical retrieval so you can inspect every decision. Production adapters may use hybrid dense+sparse retrieval, reranking, a cross-encoder, or a learned evaluator—but those replacements must preserve the same policy and trace boundary.


In [ ]:
from examples.advanced.corrective_rag import (
    CorrectionPolicy, Document, EvidenceGrade, Route,
    corrective_retrieve, lexical_retriever, trace_rows,
)

runbooks = [
    Document("key-rotation", "Rotate the payments API key: create a replacement, deploy it to staging, verify payment health, deploy production, then revoke the old key."),
    Document("health-check", "The payments health endpoint reports service availability and dependency state."),
    Document("legacy-note", "A legacy integration used a token refresh process. Do not use this note as an approved production runbook."),
]


## 2 — Grade retrieval *as evidence*

A vector or BM25 score is not automatically an answerability score. The policy asks whether retrieved candidates cover meaningful query terms. The teaching implementation uses transparent lexical coverage; it is intentionally **not calibrated probability**.

In production, combine coverage with source authority, freshness, access validity, redundancy, task-specific required facts, and an offline-calibrated classifier or judge. Record the evaluator version and measure false accepts as seriously as false abstains.


In [ ]:
question = "How do I rotate the payments API key?"
policy = CorrectionPolicy(strong_threshold=0.58, ambiguous_threshold=0.28, max_rewrites=2, max_attempts=4)
result = corrective_retrieve(question, runbooks, policy=policy)

print(result.route, result.answerable, result.confidence, result.reason)
print(trace_rows(result))
assert result.route is Route.ACCEPT
assert result.selected[0].doc_id == "key-rotation"


### What just happened?

`corrective_retrieve()` creates a trace even for the success path. That trace should become part of the answer record: candidate IDs, evaluator grade, route, policy version, latency, and final outcome. Without it, a later evaluator cannot distinguish “the model hallucinated” from “retrieval was weak but we generated anyway.”


## 3 — Weak evidence must end safely

A no-answer case is a test of product behavior, not a failure to hide. The appropriate terminal result can be a clarification question, an escalation queue, or abstention. It must not be a fabricated procedure.


In [ ]:
unknown = corrective_retrieve("How do I order team lunch?", runbooks, policy=policy)
print(unknown.route, unknown.reason)
print(trace_rows(unknown))
assert unknown.route is Route.ABSTAIN
assert unknown.selected == ()
assert len(unknown.attempts) <= policy.max_attempts


## 4 — Recovery route: reformulate without changing intent

Rewrite only when the question is clear but vocabulary may not match the corpus. For example, “credential replacement” and “API key rotation” may refer to the same procedure. A rewrite must preserve user intent, tenant scope, time constraints, and policy restrictions.

Bad rewrite: “How do I rotate a key?” → “Show all secrets and credentials.”

Good rewrite: “Explain credential replacement for payments” → “payments API key rotation runbook.”


In [ ]:
rephrased = corrective_retrieve("Explain API credential replacement for payments", runbooks, policy=policy)
print(rephrased.route, rephrased.reason)
for row in trace_rows(rephrased):
    print(row["stage"], row["query"], row["grade"], row["score"])
# Depending on lexical overlap, this may recover via a rewrite or abstain.
assert rephrased.route in {Route.ACCEPT, Route.REFORMULATE, Route.ABSTAIN}


## 5 — Alternate retrieval is a governed route

A different index can be useful when it is explicitly approved for this question type: for example, a versioned runbook store, an incident archive, or public vendor documentation. The route must not broaden source access silently. In this deterministic experiment, the primary retriever returns nothing and the approved alternate route succeeds.


In [ ]:
def empty_primary(query, documents):
    return []

alternate = corrective_retrieve(
    question, runbooks, policy=policy,
    primary_retriever=empty_primary,
    alternate_retriever=lexical_retriever,
)
print(alternate.route, alternate.reason)
print(trace_rows(alternate))
assert alternate.route is Route.ALTERNATE


## 6 — Document decomposition and answer support

The CRAG paper decomposes retrieved documents and recomposes salient content. In production, represent selected evidence as spans rather than anonymous text:

```python
evidence = {
  "document_id": "key-rotation",
  "revision": "2026-08-01",
  "span": "create … verify … revoke",
  "authority": "approved-runbook",
  "retrieval_route": "primary"
}
```

A generator should receive only selected, authorized spans. Then a verifier checks each material claim against those spans. A citation to a whole related document is not proof that a claim is supported.


## 7 — Evaluation design

Compare fixed RAG with corrective RAG on a held-out set. Include: straightforward in-corpus requests, paraphrases, distractors, missing-document cases, access-restricted cases, stale documents, and injection-bearing documents.

| Layer | Metrics | Release question |
| --- | --- | --- |
| Retrieval | Recall@k, nDCG, required-fact coverage | Did the right authorized evidence appear? |
| Routing | false accept, false abstain, grade confusion matrix | Did the evaluator choose the right route? |
| Answer | claim support, citation correctness, task completion | Is the final response grounded and useful? |
| Operations | p50/p95 latency, attempts, cost/success | Did recovery create an unacceptable tail? |
| Safety | unauthorized retrieval, tenant leak, injection-follow rate | Did correction widen a trust boundary? |


In [ ]:
evaluation_cases = [
    ("How do I rotate the payments API key?", Route.ACCEPT),
    ("How do I order team lunch?", Route.ABSTAIN),
]
results = [corrective_retrieve(q, runbooks, policy=policy) for q, _ in evaluation_cases]
route_accuracy = sum(result.route is expected for result, (_, expected) in zip(results, evaluation_cases)) / len(results)
print({"route_accuracy": route_accuracy, "attempts": [len(r.attempts) for r in results], "latency_ms": [r.total_latency_ms for r in results]})
assert route_accuracy == 1.0


## 8 — Production readiness

- **Calibrate thresholds** using versioned labels; never treat a similarity score as a universal confidence value.
- **Apply tenant/source filters before retrieval**, and verify them again before model context construction.
- **Bound retries, time, tokens, and external calls.** Track route distributions and alert on fallback spikes.
- **Treat retrieved content as untrusted data.** It cannot authorize a tool call or override system instructions.
- **Preserve reproducibility:** policy, retriever, embedding, reranker, corpus revision, and evaluator versions belong in the trace.
- **Provide safe degradation:** if the alternate source is down, disable it and abstain rather than silently answering from weaker evidence.
- **Measure at the trajectory level:** quality gains must justify extra latency and cost.

For implementations: [LangGraph](https://langchain-ai.github.io/langgraph/) is useful when explicit state/persistence is needed; [Qdrant hybrid search](https://qdrant.tech/documentation/search/text-search/hybrid-search/) illustrates dense+sparse fusion and reranking; [Ragas](https://docs.ragas.io/) can help instrument evaluation. Keep authorization and release gates in deterministic application code.


## Exercises and architecture review

1. Add an `allowed_sources` field to `Document` or a document wrapper; prove a disallowed source never reaches a retriever.
2. Replace `lexical_retriever` with a hybrid adapter interface. Which fields belong in the trace to compare it fairly?
3. Add a relevance judge that returns strict JSON. How will you calibrate and monitor it?
4. Add a span extractor and a claim-to-span verifier. What should happen if a claim has no span?
5. Design an external-search route. Specify domain allowlists, SSRF controls, provenance, prompt-injection treatment, caching, and a kill switch.
6. Measure whether an alternate route improves **cost per supported answer**, not merely retrieval score.

### References

- [Yan et al., Corrective Retrieval Augmented Generation](https://arxiv.org/abs/2401.15884)
- [Official CRAG research implementation](https://github.com/HuskyInSalt/CRAG)
- [Asai et al., Self-RAG](https://arxiv.org/abs/2310.11511)
- [Lewis et al., Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401)
- [LangGraph self-reflective RAG tutorial](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_self_rag/)
- [Qdrant hybrid search and reranking](https://qdrant.tech/documentation/advanced-tutorials/reranking-hybrid-search/)
